# Capítulo 9: k-Vizinhos Mais Próximos

**Bases 5 — Ciência de Dados** · notebook de aula

Cada célula de código é a mesma do livro e roda na ordem em que aparece — execute de cima para baixo. Versão publicada deste capítulo: [https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/index.html](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/index.html)

> **Gerado automaticamente a partir dos `.qmd` do livro por `scripts/gerar-notebooks.py`.** Edições feitas aqui se perdem no próximo `make notebooks`; para mudar o conteúdo, edite o `.qmd`.

In [ ]:
# Põe o diretório de trabalho na raiz do projeto — é o que faz
# `from scratch...` e os caminhos `dados/...` funcionarem. No livro isso vem
# do `execute-dir: project` do Quarto; aqui é feito à mão.
#
# No Colab não existe cópia do projeto, então esta célula clona uma. É rápido
# (clone raso) e acontece só na primeira execução da sessão.
import os
import subprocess
import sys

REPO = "https://github.com/BragaD/UnDF-Bases5-CienciaDeDados-202602.git"


def raiz_do_projeto(inicio="."):
    """Sobe os diretórios até achar o `_quarto.yml`. None se não houver."""
    atual = os.path.abspath(inicio)
    while not os.path.exists(os.path.join(atual, "_quarto.yml")):
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai
    return atual


raiz = raiz_do_projeto()
if raiz is None:
    destino = "/content/bases5" if os.path.isdir("/content") else "bases5"
    if not os.path.isdir(destino):
        print("baixando o material da disciplina...")
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    raiz = raiz_do_projeto(destino)

os.chdir(raiz)
if raiz not in sys.path:
    sys.path.insert(0, raiz)

%matplotlib inline
print("diretório de trabalho:", os.getcwd())

> **📌 Nota**
>
> Este capítulo corresponde ao capítulo 12 de Grus (2019).

> Se você quiser irritar os seus vizinhos, conte a verdade sobre eles.
>
> — Pietro Aretino

Quase todo modelo deste livro olha o conjunto de dados inteiro para aprender um padrão: ajusta coeficientes, mede erros, itera. O k-vizinhos mais próximos não faz nada disso. Ele não aprende — ele guarda. Na hora de classificar um ponto novo, procura os pontos rotulados mais parecidos e deixa que eles votem.

É o modelo mais simples deste livro. Os capítulos 5 e 8 já fizeram você implementar peças do zero — gradiente descendente, funções de avaliação —, mas este é o primeiro *classificador* completo que você monta do dado bruto até a previsão. Ele precisa de exatamente duas coisas: uma noção de distância, e a hipótese de que pontos próximos se parecem.

Ao final deste capítulo, você será capaz de:

- Explicar o que o k-vizinhos faz e o que ele deliberadamente ignora
- Implementar uma votação majoritária que resolve empates de forma determinística
- Classificar dados reais com o algoritmo que você escreveu
- Explicar por que aumentar o número de dimensões degrada o método, e demonstrar isso numericamente
- Reconhecer o mesmo algoritmo na interface do `scikit-learn`

## Seções

| Seção | Tópico |
|---|---|
| [9.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/01-o-modelo.html) | O Modelo |
| [9.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/02-exemplo-o-dataset-iris.html) | Exemplo: O Dataset Iris |
| [9.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/03-a-maldicao-da-dimensionalidade.html) | A Maldição da Dimensionalidade |

## O Modelo

> **📌 Nota**
>
> Esta seção corresponde a *The Model*, do capítulo 12 de Grus (2019).

Imagine que você quer prever em quem uma pessoa vai votar. Sem saber mais nada sobre ela, uma aposta razoável é olhar como os vizinhos dela votam. Se você souber mais — idade, renda, quantos filhos tem —, dá para olhar os vizinhos *naquelas dimensões* também, e não só na geográfica.

Essa é a ideia inteira.

> **🔷 Conceito**
>
> O k-vizinhos mais próximos precisa de apenas duas coisas:
>
> 1. Uma noção de **distância**
> 2. A hipótese de que pontos **próximos são parecidos**
>
> Ele não faz nenhuma suposição matemática sobre a forma dos dados e não tem etapa de treino. Em compensação, **ignora quase toda a informação disponível**: a previsão para um ponto novo depende só do punhado de pontos mais próximos dele.

Essa é uma troca real, não um detalhe. O k-vizinhos raramente ajuda a *entender* o fenômeno. Prever o voto de alguém a partir do voto dos vizinhos não diz nada sobre a causa daquele voto — enquanto um modelo baseado em renda e estado civil poderia dizer.

### Contando votos

Escolhido um *k* — 3 ou 5, digamos —, classificar um ponto novo é achar os *k* pontos rotulados mais próximos e deixá-los votar. Precisamos, então, de uma função que conte votos.

A primeira tentativa é direta:

In [ ]:
import numpy as np

def raw_majority_vote(labels) -> str:
    valores, contagens = np.unique(labels, return_counts=True)
    return valores[np.argmax(contagens)]

assert raw_majority_vote(['a', 'b', 'c', 'b']) == 'b'

Só que ela não faz nada de inteligente com empates. `np.unique` devolve os rótulos em ordem alfabética e as contagens na mesma ordem; `np.argmax` devolve a posição do maior valor — e, no empate, a do **primeiro**. Num empate, portanto, vence o rótulo alfabeticamente menor — e nada avisa que houve empate. Se estivéssemos classificando filmes e os cinco mais próximos fossem G, G, PG, PG e R, teríamos dois votos para G e dois para PG. Há três saídas possíveis:

- Escolher um dos vencedores ao acaso
- Ponderar os votos pela distância e pegar o vencedor ponderado
- Reduzir *k* até haver um vencedor único

Vamos implementar a terceira:

In [ ]:
def majority_vote(labels) -> str:
    """Assume que os rótulos estão ordenados do mais próximo ao mais distante."""
    valores, contagens = np.unique(labels, return_counts=True)
    vencedores = valores[contagens == contagens.max()]

    if len(vencedores) == 1:
        return vencedores[0]                # vencedor único
    else:
        return majority_vote(labels[:-1])   # tenta de novo sem o mais distante

# Empate: olha os 4 primeiros, então 'b'
assert majority_vote(['a', 'b', 'c', 'b', 'a']) == 'b'

> **🟩 Exemplo**
>
> `contagens == contagens.max()` é a máscara booleana da [seção 7.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/01-explorando-seus-dados.html): ela marca todos os rótulos empatados no topo, e o tamanho da seleção é o que diz se houve empate. A recursão sempre termina. No pior caso, descartamos um rótulo por vez até sobrar um só — e aí ele vence sozinho.
>
> Repare que o argumento precisa estar **ordenado do mais próximo ao mais distante**. Descartar "o último" só faz sentido se o último for o vizinho menos relevante. Uma sequência fora de ordem produz uma resposta errada sem erro nenhum.

### O classificador

Com a votação pronta, o classificador cabe em poucas linhas. A distância que o [Capítulo 4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap04/01-vetores.html) escreveu como um `sum` sobre um `zip` é agora `np.linalg.norm` — e ela não mede uma distância de cada vez. `X_train - new_point` subtrai o ponto novo de **cada linha** da matriz de treino (o broadcasting da [seção 7.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/01-explorando-seus-dados.html)), e `axis=1` reduz cada linha do resultado a um número: as $n$ distâncias saem numa chamada só. `np.argsort` devolve os **índices** que ordenariam esse vetor, e é com eles que escolhemos os $k$ rótulos mais próximos:

In [ ]:
def knn_classify(k: int,
                 X_train: np.ndarray,
                 y_train: np.ndarray,
                 new_point: np.ndarray) -> str:
    # A distância do ponto novo a cada linha de X_train, de uma vez.
    dists = np.linalg.norm(X_train - new_point, axis=1)

    # Os índices que ordenam os pontos do mais próximo ao mais distante...
    ordem = np.argsort(dists)

    # ...e os rótulos dos k primeiros votam.
    return majority_vote(y_train[ordem[:k]])

# Dois pontos perto da origem, três longe dela.
X_exemplo = np.array([[0.0, 0.0], [0.0, 1.0], [3.0, 3.0], [3.0, 4.0], [4.0, 3.0]])
y_exemplo = np.array(['perto', 'perto', 'longe', 'longe', 'longe'])

assert knn_classify(3, X_exemplo, y_exemplo, np.array([0.5, 0.5])) == 'perto'
assert knn_classify(3, X_exemplo, y_exemplo, np.array([3.5, 3.5])) == 'longe'

É isso: medir todas as distâncias, ordenar, cortar em *k*, contar. Não há treino, não há parâmetros ajustados, não há otimização. O modelo *é* o conjunto de dados.

> **💡 Dica — Na prática: `scikit-learn`**
>
> Você acabou de escrever o algoritmo. Na vida real, você usaria isto:
>
> ```python
> from sklearn.neighbors import KNeighborsClassifier
>
> modelo = KNeighborsClassifier(n_neighbors=5)
> modelo.fit(X_treino, y_treino)
> modelo.predict(X_novo)
> ```
>
> Sobre o custo: as $n$ distâncias saem numa chamada só, mas o trabalho continua sendo $O(n \cdot d)$ — o que o `numpy` tirou foi o interpretador do caminho, não a conta. Depois disso, `np.argsort` ordena **todas** as $n$ distâncias, a $O(n \log n)$, para usar só as $k$ primeiras. Não precisávamos ordenar tudo: `np.argpartition(dists, k)` separa os $k$ menores do resto em $O(n)$, sem ordenar nada, e bastaria ordenar esses $k$ depois — a ordem entre eles importa, porque é ela que o `majority_vote` consome ao descartar o mais distante no empate. Não fizemos isso porque o objetivo aqui era clareza, não desempenho — mas é a otimização óbvia se este código fosse para produção.
>
> Sobre o desempate: o `scikit-learn` não reduz *k* como fizemos. Ele toma a moda dos rótulos codificados, e como o atributo `classes_` fica ordenado, o desempate favorece o rótulo alfabeticamente menor — exatamente o que o `np.argmax` do `raw_majority_vote` fazia, e que descartamos. Não é a regra da nossa recursão, mas é igualmente arbitrária. Ele também aceita pesos por distância (`weights='distance'`).
>
> Sobre a busca: por padrão (`algorithm='auto'`), o `scikit-learn` escolhe entre força bruta e estruturas de indexação como `KDTree` ou `BallTree`, que evitam comparar o ponto novo com todos os outros. Mas essas estruturas só ajudam em dimensão baixa — em dimensão alta elas degradam para força bruta, o mesmo custo que o nosso código sempre teve. O motivo é o assunto da seção 9.3: a partir de um certo número de dimensões, nem a distância nem a estrutura que a indexa continuam ajudando.
>
> Nada disso muda o que o modelo *é*. É a mesma ordenação por distância seguida de votação que você implementou acima.

## Exemplo: O Dataset Iris

> **📌 Nota**
>
> Esta seção corresponde a *Example: The Iris Dataset*, do capítulo 12 de Grus (2019).

O *Iris* é um clássico do aprendizado de máquina. São 150 flores de três espécies, e para cada uma temos quatro medidas: comprimento e largura da pétala, comprimento e largura da sépala. A tarefa é prever a espécie a partir das quatro medidas.

A fonte original é o repositório da UCI:

```python
import requests

data = requests.get(
  "https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data"
)

with open('iris.dat', 'w') as f:
    f.write(data.text)
```

> **❗ Importante**
>
> Esse download **não é executado** aqui: o arquivo já está salvo em `dados/iris.data`, e é dele que vamos ler.
>
> Uma análise que baixa o dado de novo a cada execução é frágil. Basta o servidor sair do ar, mudar de endereço ou passar a servir uma versão diferente do arquivo para o resultado deixar de ser reproduzível — e você não tem como saber qual das três coisas aconteceu. Baixe uma vez, guarde a cópia, trabalhe em cima dela. O código do download continua valendo a pena registrar, porque saber de onde o dado veio *é* parte da análise.

Os dados são separados por vírgula, com os campos:

```
sepal_length, sepal_width, petal_length, petal_width, class
```

A primeira linha, por exemplo:

```
5.1,3.5,1.4,0.2,Iris-setosa
```

### Carregando os dados

Nossa função de vizinhos espera duas coisas: uma matriz de medidas, com uma linha por flor, e um array com o rótulo de cada linha. É assim que vamos carregar o arquivo — e são duas leituras, não uma, porque um array tem **um** tipo só: as quatro medidas são `float`, a espécie é texto — e com `dtype=str` o mesmo `np.loadtxt` que a [seção 6.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/01-lendo-arquivos.html) viu recusar texto lê agora só a coluna de texto. `np.char.replace` aplica o `.replace` de string a cada elemento do array.

In [ ]:
import numpy as np
from scratch_np.k_nearest_neighbors import knn_classify

# As quatro medidas: uma linha por flor, uma coluna por medida.
X = np.loadtxt("dados/iris.data", delimiter=",", usecols=range(4))

# A espécie vem como "Iris-virginica"; queremos só "virginica".
rotulos = np.loadtxt("dados/iris.data", delimiter=",", usecols=4, dtype=str)
y = np.char.replace(rotulos, "Iris-", "")

especies = np.unique(y)

X.shape, especies

> **📌 Nota**
>
> Dois detalhes do carregamento:
>
> - O arquivo termina com uma **linha em branco** — arquivo de dado real quase sempre tem uma dessas — e quem a descarta é o `np.loadtxt`: linha vazia não vira linha do array, e por isso `X.shape` sai `(150, 4)`, não `(151, 4)`. O Grus (2019), que lê com o módulo `csv`, precisa de um `if row` explícito para o mesmo efeito. O que continua sendo seu é **conferir a forma** — e repare que a tolerância do `np.loadtxt` acaba aí: um campo a menos, ou texto onde deveria haver número, interrompe a leitura com `ValueError`, que é o leitor que reclama recomendado pela [seção 6.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/01-lendo-arquivos.html).
> - `knn_classify` não é redefinida aqui: vem de `scratch_np.k_nearest_neighbors`, o mesmo código que você escreveu na [seção anterior](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/01-o-modelo.html).

### Olhando os dados

Gostaríamos de visualizar as medidas para ver como variam por espécie. O problema é que são quatro dimensões, o que dificulta o desenho. Uma saída é olhar os gráficos de dispersão para cada um dos seis pares de medidas. Não é preciso agrupar as flores por espécie antes: `y == especie` é uma máscara, e `X[desta, i]` é a coluna *i* só das linhas daquela espécie.

In [ ]:
# Figura: Dispersões do Iris, para os seis pares de medidas
from matplotlib import pyplot as plt

metrics = ['comprimento sépala', 'largura sépala',
           'comprimento pétala', 'largura pétala']
pairs = [(i, j) for i in range(4) for j in range(4) if i < j]
marks = ['+', '.', 'x']  # três classes, três marcadores

fig, ax = plt.subplots(2, 3, figsize=(10, 6))

for row in range(2):
    for col in range(3):
        i, j = pairs[3 * row + col]
        ax[row][col].set_title(f"{metrics[i]} vs {metrics[j]}", fontsize=8)
        ax[row][col].set_xticks([])
        ax[row][col].set_yticks([])

        for mark, especie in zip(marks, especies):
            desta = y == especie                  # máscara: quais linhas são desta espécie
            ax[row][col].scatter(X[desta, i], X[desta, j],
                                 marker=mark, label=especie)

ax[-1][-1].legend(loc='lower right', prop={'size': 6})
plt.tight_layout()
plt.show()

As medidas realmente se agrupam por espécie. Olhando só para as sépalas, seria difícil separar *versicolor* de *virginica* — mas quando entram comprimento e largura da pétala, a separação fica clara. É exatamente a situação em que vizinhos mais próximos funciona bem.

### Classificando

Primeiro dividimos os dados em treino e teste:

In [ ]:
from scratch_np.machine_learning import train_test_split

rng = np.random.default_rng(12)
X_train, X_test, y_train, y_test = train_test_split(X, y, 0.30, rng)

assert len(X_train) == 0.7 * 150
assert len(X_test) == 0.3 * 150

len(X_train), len(X_test)

> **⚠️ Atenção — A semente não é opcional**
>
> `np.random.default_rng(12)` cria o gerador que o `train_test_split` usa para sortear a divisão — e ele é **parâmetro**, não estado global: a semente fica escrita aqui, ao lado da divisão que produziu. Sem ela, cada execução separa um conjunto de treino diferente — e com ele vêm uma acurácia diferente e uma matriz de confusão diferente. Você não conseguiria repetir o próprio resultado, nem comparar duas escolhas de *k* sabendo que a diferença veio do *k* e não do sorteio.
>
> Fixe a semente em todo experimento que usa aleatoriedade. O `train_test_split` da [seção 8.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/03-overfitting-e-underfitting.html) não tem valor padrão para o gerador, justamente para que ninguém consiga esquecê-lo.

Os pontos de treino são os "vizinhos" que usaremos para classificar os pontos de teste. Falta escolher *k*. Pequeno demais (pense em *k* = 1) e os outliers têm influência exagerada; grande demais (pense em *k* = 105) e simplesmente prevemos a classe mais comum do conjunto. Numa aplicação real criaríamos um conjunto de validação para escolher; aqui vamos usar *k* = 5:

In [ ]:
y_pred = np.array([knn_classify(5, X_train, y_train, ponto)
                   for ponto in X_test])

acertos = y_pred == y_test        # uma máscara com 45 booleanos
pct_correct = float(acertos.mean())   # float(): um número do Python, não um escalar do numpy
pct_correct

O laço percorre as 45 flores de teste, não os 105 vizinhos de cada uma: a conta pesada — a distância de um ponto a todo o conjunto de treino — já é uma operação de array dentro do `knn_classify`. O que sobrou no laço é a votação, que é o algoritmo. Depois dele, `y_pred == y_test` compara os 45 pares de uma vez, e `acertos.mean()` é a fração de acertos, porque `True` conta como 1.

In [ ]:
previsto = np.searchsorted(especies, y_pred)   # 0, 1 ou 2: a linha
real = np.searchsorted(especies, y_test)       # 0, 1 ou 2: a coluna

confusion_matrix = np.zeros((len(especies), len(especies)), dtype=int)
np.add.at(confusion_matrix, (previsto, real), 1)

titulo = "previsto \\ real"
print(f"{titulo:>18}" + "".join(f"{especie:>12}" for especie in especies))
for i, especie in enumerate(especies):
    print(f"{especie:>18}" + "".join(f"{n:>12d}" for n in confusion_matrix[i]))

> **🟩 Exemplo**
>
> `np.searchsorted(especies, y_pred)` troca cada rótulo pelo índice dele em `especies` — que o `np.unique` devolveu em ordem alfabética —, e o par (previsto, real) é a posição na matriz.
>
> O `np.add.at` merece atenção. O que se escreveria naturalmente, `confusion_matrix[previsto, real] += 1`, **não funciona** com arrays de índices: o `numpy` lê os valores antigos, soma 1 e grava o resultado de uma vez, então um par que aparece dezessete vezes é gravado como 1. O `np.add.at` soma no lugar, acumulando as repetições. É mais um erro que não levanta exceção — a matriz sai, só sai errada.

A diagonal são os acertos: a linha diz o que o modelo respondeu, a coluna diz o que a flor era. Neste conjunto simples, o modelo acerta quase tudo. Há uma *versicolor* classificada como *virginica* — justamente o par que os gráficos de dispersão mostraram ser o mais difícil de separar — e o resto sai certo.

> **💡 Dica — Na prática: `scikit-learn`**
>
> O mesmo experimento, com a biblioteca:
>
> ```python
> from sklearn.model_selection import train_test_split
> from sklearn.neighbors import KNeighborsClassifier
> from sklearn.metrics import confusion_matrix, accuracy_score
>
> X_treino, X_teste, y_treino, y_teste = train_test_split(
>     X, y, test_size=0.3, random_state=12
> )
>
> modelo = KNeighborsClassifier(n_neighbors=5).fit(X_treino, y_treino)
> previsto = modelo.predict(X_teste)
>
> accuracy_score(y_teste, previsto)
> confusion_matrix(y_teste, previsto)
> ```
>
> Repare no que **não** há aqui: nenhuma linha para montar `X` e `y`. Eles já são o que a biblioteca espera — uma matriz `(150, 4)` e um vetor de 150 rótulos —, porque a convenção de array deste livro é a mesma dela.
>
> A divisão, porém, não sai idêntica — e a matriz de confusão tampouco, com outros totais por espécie no teste: `random_state=12` alimenta o gerador do `scikit-learn`, que não é o `np.random.default_rng(12)` daqui. Sementes iguais em geradores diferentes dão sorteios diferentes. Que a acurácia saia a mesma (44 em 45) é coincidência, não confirmação — sem contar o desempate, que também é outro. Repare ainda na orientação: `confusion_matrix(y_teste, previsto)` traz o **real** nas linhas e o previsto nas colunas — a transposta da nossa tabela previsto × real.
>
> Neste dataset — 150 pontos, 4 dimensões — o que a biblioteca economiza é digitação, não conta: as 45 classificações do nosso laço levam menos de um centésimo de segundo, cada uma resolvida por uma subtração e uma norma sobre uma matriz de 105 × 4. A seção anterior mostrou onde o `scikit-learn` realmente ganha: indexação, pesos por distância, escala — e o `predict` dele classifica os 45 pontos de uma vez, em vez de um por um como o nosso laço. A vantagem de verdade aparece quando os dados crescem, não neste exemplo.

## A Maldição da Dimensionalidade

> **📌 Nota**
>
> Esta seção corresponde a *The Curse of Dimensionality*, do capítulo 12 de Grus (2019).

O k-vizinhos tem um problema sério em dimensões altas, e o nome dele é **maldição da dimensionalidade**. A raiz é simples de enunciar: espaços de dimensão alta são *vastos*. Pontos neles tendem a não estar perto de ninguém.

Dá para ver isso experimentalmente. Vamos gerar pares de pontos aleatórios num "cubo unitário" de dimensão *d*, para vários valores de *d*, e medir as distâncias.

Gerar pontos aleatórios agora é uma linha: `rng.random((num_pairs, dim))` devolve uma matriz inteira de coordenadas sorteadas em $[0, 1)$ — uma linha por ponto, uma coluna por dimensão. Duas dessas matrizes são os dois lados de `num_pairs` pares; a subtração casa linha com linha, e `axis=1` reduz cada uma a uma distância:

In [ ]:
import numpy as np

def random_distances(dim: int, num_pairs: int,
                     rng: np.random.Generator) -> np.ndarray:
    """Distâncias de `num_pairs` pares sorteados no cubo unitário de dimensão `dim`."""
    return np.linalg.norm(rng.random((num_pairs, dim))
                          - rng.random((num_pairs, dim)), axis=1)

`random_distances` fica em `scratch_np.k_nearest_neighbors`, ao lado do classificador da [seção 9.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/01-o-modelo.html).

Para cada dimensão de 1 a 100, calculamos 10.000 distâncias e guardamos a média e a mínima:

In [ ]:
# Figura: Distância média e mínima entre pontos aleatórios, por dimensão
from matplotlib import pyplot as plt

dimensions = range(1, 101)
rng = np.random.default_rng(0)

avg_distances = []
min_distances = []

for dim in dimensions:
    distances = random_distances(dim, 10000, rng)   # 10.000 pares de uma vez
    avg_distances.append(distances.mean())
    min_distances.append(distances.min())

avg_distances = np.array(avg_distances)
min_distances = np.array(min_distances)

plt.plot(dimensions, avg_distances, label='distância média')
plt.plot(dimensions, min_distances, label='distância mínima')
plt.xlabel("# de dimensões")
plt.ylabel("distância")
plt.title("10.000 distâncias aleatórias")
plt.legend()
plt.show()

São um milhão de distâncias euclidianas — 10.000 pares para cada uma das 100 dimensões —, mas o laço que sobrou percorre as **dimensões**, não os pares: cada iteração é uma chamada de `np.linalg.norm` sobre uma matriz de 10.000 linhas, e o experimento inteiro termina em um ou dois segundos. Não há progresso que valha uma barra de `tqdm` ([seção 7.7](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/07-um-parenteses-tqdm.html)) para acompanhar. O gerador é criado uma vez, com `np.random.default_rng(0)`, e atravessa as 100 chamadas: é o que garante que a figura seja a mesma toda vez que o experimento é repetido.

A distância média entre dois pontos cresce com a dimensão, o que já era de esperar. O que incomoda é outra coisa: a **razão** entre a menor distância e a distância média.

In [ ]:
# Figura: Razão entre a menor distância e a distância média
min_avg_ratio = min_distances / avg_distances

plt.plot(dimensions, min_avg_ratio)
plt.xlabel("# de dimensões")
plt.ylabel("razão")
plt.title("Distância mínima / distância média")
plt.show()

> **🔷 Conceito**
>
> Em dimensão baixa, o ponto mais próximo está **muito** mais perto que a média — a razão fica perto de zero, e "vizinho mais próximo" significa alguma coisa.
>
> Conforme a dimensão cresce, a razão sobe assintoticamente em direção a 1: o ponto mais próximo fica cada vez mais parecido, em distância, com um ponto qualquer. No gráfico acima, com *d* indo só até 100, ela já ronda 0,75 — em *d* = 100 dá 0,75 — e a curva ainda sobe, agora em zigue-zague: a **menor** de 10.000 distâncias é uma estatística instável, e basta um par especialmente próximo para o mínimo daquela dimensão despencar. Ainda não chegou perto de 1, mas a tendência é clara, e ela seguiria subindo se a dimensão continuasse crescendo. É esse limite, não o valor em *d* = 100, que importa: quando a razão está perto de 1, "vizinho mais próximo" deixa de significar coisa alguma.

A intuição por trás disso: dois pontos só estão próximos se estiverem próximos em **todas** as dimensões. Cada dimensão extra — mesmo que seja puro ruído — é mais uma oportunidade para dois pontos ficarem distantes um do outro. Com dimensão suficiente, todo mundo fica longe de todo mundo.

A ressalva importa: isso vale a menos que haja muita estrutura nos dados que os faça se comportar como se tivessem dimensão bem menor. Um conjunto de 100 colunas em que 97 são combinações lineares das outras 3 não é, de fato, um conjunto de 100 dimensões.

> **💡 Dica — Na prática: o que se faz com isso**
>
> Quando os dados têm dimensão alta demais para vizinhos mais próximos, as saídas usuais são reduzir a dimensão antes de classificar — PCA, que aparece no capítulo 7, é a mais comum — ou trocar por um modelo que não dependa de distância, como as árvores de decisão do capítulo 14.
>
> O `scikit-learn` não protege você disso. `KNeighborsClassifier` aceita 500 colunas sem reclamar, roda, devolve previsões, e elas serão ruins por um motivo que nenhuma mensagem de erro vai explicar. Saber *por quê* é o que você leva desta seção.

## Leituras adicionais

O `scikit-learn` traz muitos [modelos de vizinhos mais próximos](https://scikit-learn.org/stable/modules/neighbors.html), incluindo variantes com pesos por distância e estruturas de indexação (`KDTree`, `BallTree`) que evitam comparar o ponto novo com todos os outros.

Para o tratamento estatístico do compromisso entre viés e variância na escolha de *k*, veja Hastie et al. (2009).

## Referências

- **Grus**. *Data Science from Scratch: First Principles with Python*. 2nd ed.. O'Reilly Media. 2019.
- **Hastie; Tibshirani; Friedman**. *The Elements of Statistical Learning*. 2nd ed.. Springer. 2009.